# Data Wrangling - Part 1: Selecting Rows and Handling Missing Values

## Why data wrangling matters

Data wrangling transforms raw observations into analysis-ready data. Environmental datasets often contain multiple sites, sampling dates, inconsistent values, and missing measurements that must be filtered or cleaned before analysis.

In this lesson you will:
- inspect the values in a dataset before filtering
- select rows using Boolean conditions
- combine multiple filtering criteria
- identify and handle missing values.

## Objectives
By the end of this lesson, you should be able to:

1. Describe the role of data wrangling in an environmental data analysis workflow
2. Select rows from a pandas DataFrame using Boolean masks and `.loc[]`
3. Select rows from a pandas DataFrame using `.query()`
4. Combine multiple filtering criteria using comparison, logical, and membership operators
5. Identify, quantify, and filter missing values
6. Apply these techniques to create a small analysis-ready subset of an environmental dataset


## Set up your session

We will use a lake monitoring dataset from the North Temperate Lakes Long-Term Ecological Research (NTL-LTER) program. It contains repeated measurements collected from multiple lakes over many years.

> 👉*Before continuing:* Familiarize yourself the NTL-LTER dataset and why long-term monitoring is valuable.

In [ ]:
# Packages
from pathlib import Path
import pandas as pd

In [ ]:
# Paths
root_fldr = Path.cwd().parent
raw_fldr = root_fldr / "data" / "raw"
processed_fldr = root_fldr / "data" / "processed"

raw_fldr

In [ ]:
# Data import
NTL_phys_data = pd.read_csv(
    raw_fldr / "NTL-LTER_Lake_ChemistryPhysics_Raw.csv",
    dtype={
        "lakeid": "category",
        "lakename": "category",
    },
    parse_dates=["sampledate"],
    date_format="%m/%d/%y"
)

In [ ]:
# Preview the first few rows
NTL_phys_data.head()

In [ ]:
# Show dataframe structure, data types, and non-null counts
NTL_phys_data.info()

In [ ]:
# Quick summary statistics for numeric columns
NTL_phys_data.describe()

---
## Inspect values before filtering

Before filtering a dataset, it is good practice to inspect the values you might filter on. This helps prevent mistakes caused by spelling, capitalization, abbreviations, or unexpected categories.


In [ ]:
# What lakes are represented in the dataset?
NTL_phys_data["lakename"].unique()

In [ ]:
# 🔥Since `lakename` is a category field, we can use its `cat.categories` property to list unique values  
NTL_phys_data['lakename'].cat.categories

In [ ]:
# How many observations are available for each lake?
NTL_phys_data["lakename"].value_counts()

👉 *How might you display this graphically?*

In [ ]:
# Plot how many observations are available for each lake

In [ ]:
# What is the distribution of temperature values?
NTL_phys_data["temperature_C"].describe()

👉 *How might you display this graphically?*

In [ ]:
# Plot the distribution of depth values


---
## ◾Subsetting data: rows

Suppose we only want to work with specific observations in the northern temperate lakes dataset. For example, we may only want data collected at the surface, from a specific lake, or from a selected set of lakes.

There are two common ways to select rows in pandas:

- `.loc[]` with a **Boolean mask**
- `.query()` with a **query string**


### Selecting rows using `.loc[]` and Boolean masks

A **Boolean mask** is a Series of `True` and `False` values. Each value indicates whether the corresponding row meets a condition.

When a Boolean mask is passed to `.loc[]`, pandas returns only the rows where the mask is `True`.


In [ ]:
# Create a Boolean mask for surface records where depth equals 0
surface_mask = NTL_phys_data["depth"] == 0

In [ ]:
# Inspect the mask itself
surface_mask.head(10)

In [ ]:
# Count how many rows meet the condition
surface_mask.sum()

In [ ]:
# Apply the mask to create a dataframe of surface records
NTL_surface_loc = NTL_phys_data.loc[surface_mask]
NTL_surface_loc.head()

Notice that the index values in the filtered dataframe are not necessarily sequential. The original row labels were preserved when the rows were filtered.

If you want a fresh sequential index, use `.reset_index(drop=True)`.


In [ ]:
# Reset the index after filtering
NTL_surface_loc_reset = NTL_surface_loc.reset_index(drop=True)
NTL_surface_loc_reset.head()

### Selecting rows using `.query()`

A **query string** is a text expression that describes the filtering condition. The expression is passed to `.query()`.

Many people find `.query()` readable, especially when combining several conditions.


In [ ]:
# Create a query string for surface records
surface_query = "depth == 0"

In [ ]:
# Apply the query string
NTL_surface_query = NTL_phys_data.query(surface_query)
NTL_surface_query.head()

In [ ]:
# Confirm that both approaches return the same number of rows
NTL_surface_loc.shape, NTL_surface_query.shape

### `.loc[]` or `.query()`?

| Method | Often useful when... |
| --- | --- |
| `.loc[]` | You want explicit pandas syntax or need maximum flexibility |
| `.query()` | You want a readable expression for row filtering |
| `.loc[]` | You are building filters programmatically from existing Python objects |
| `.query()` | Your expression is close to natural language or SQL-like syntax |

Both are useful. In this course, you should be able to read and use both patterns.


---
## ◾Element-wise comparison operators

Boolean masks and query strings can use comparison operators to select records.

| Operator | Meaning |
| :---: | :--- |
| `==` | equals |
| `!=` | does not equal |
| `<` | is less than |
| `<=` | is less than or equal to |
| `>` | is greater than |
| `>=` | is greater than or equal to |


Comparison operators can be used on categorical or text variables as well as numeric variables.


In [ ]:
# Select records from Peter Lake
NTL_peter_lake = NTL_phys_data.loc[NTL_phys_data["lakename"] == "Peter Lake"]
NTL_peter_lake.shape

In [ ]:
# The same selection using query
NTL_peter_lake_query = NTL_phys_data.query('lakename == "Peter Lake"')
NTL_peter_lake_query.shape


In [ ]:
# The same selection using query
NTL_peter_lake_query = NTL_phys_data.query('lakename == "Peter Lake"')
NTL_peter_lake_query.shape

---
### 👉 Exercise 1: Filter by lake name

Use either `.loc[]` or `.query()` to answer the following questions.

1. How many records are there for Paul Lake? (answer = `10,325`)
2. How many records are for lakes other than Paul Lake? (answer = `28,289`)
3. Why might the number of observations differ among lakes?


In [ ]:
# 1. Find the number of records for Paul Lake


In [ ]:
# 2. Find the number of records that are not from Paul Lake


**Discussion:** Briefly interpret the results before continuing.

---
## ◾Logical operators for combining conditions

Logical operators allow us to combine multiple Boolean conditions.

| Operator | Meaning |
| :---: | :--- |
| `&` | and |
| `\|` | or |
| `~` | not |

When combining Boolean expressions in pandas, put each comparison inside parentheses.

Correct:

```python
(df["depth"] == 0) & (df["lakename"] == "Peter Lake")
```

Incorrect:

```python
df["depth"] == 0 & df["lakename"] == "Peter Lake"
```


In [ ]:
# Find records that are either Peter Lake or Paul Lake using a Boolean mask
peter_or_paul_mask = (
    (NTL_phys_data["lakename"] == "Peter Lake") |
    (NTL_phys_data["lakename"] == "Paul Lake")
)

peter_or_paul_mask.sum()

In [ ]:
# Apply the mask
NTL_peter_or_paul = NTL_phys_data.loc[peter_or_paul_mask]
NTL_peter_or_paul.head()

In [ ]:
# Find surface records from Peter Lake
peter_surface = NTL_phys_data.loc[
    (NTL_phys_data["lakename"] == "Peter Lake") &
    (NTL_phys_data["depth"] == 0)
]

peter_surface.shape

---
### Exercise 2: Combine conditions

Use Boolean masks to answer the following questions.

1. How many records are both from Peter Lake and Paul Lake?
2. Does that answer make sense? Why or why not?
3. How many surface records (depth is zero) are available for Paul Lake?


In [ ]:
# 1. Find records that are both Peter Lake and Paul Lake


**Discussion:** Briefly interpret the results before continuing.

In [ ]:
# 3. Find surface records for Paul Lake


### Combining conditions with `.query()`

The `.query()` method can also combine conditions. Inside a query string, use `and`, `or`, and `not` rather than `&`, `|`, and `~`.


In [ ]:
# Find records that are either Peter Lake or Paul Lake using query
NTL_peter_or_paul_query = NTL_phys_data.query(
    'lakename == "Peter Lake" or lakename == "Paul Lake"'
)

NTL_peter_or_paul_query.shape

In [ ]:
# Find surface records from Peter Lake using query
NTL_peter_surface_query = NTL_phys_data.query(
    'lakename == "Peter Lake" and depth == 0'
)

NTL_peter_surface_query.shape

---
## ◾Membership comparisons with `.isin()`

The `.isin()` method creates a Boolean mask by checking whether values are members of a list or other collection.

This is useful when you want to select observations from several categories.


In [ ]:
# Select records from Peter Lake, Paul Lake, and Tuesday Lake
selected_lakes = ["Peter Lake", "Paul Lake", "Tuesday Lake"]

NTL_three_lakes = NTL_phys_data.loc[NTL_phys_data["lakename"].isin(selected_lakes)]
NTL_three_lakes.shape

In [ ]:
# Check which lakes are present in the filtered dataset
NTL_three_lakes["lakename"].value_counts()

---
### Exercise 3: Use `.isin()`

Create a dataframe containing only observations from Peter Lake, Paul Lake, and Tuesday Lake where depth is greater than 5 meters.

Then answer:

1. How many rows are in the resulting dataframe?
2. Which lake contributes the most observations?


In [ ]:
# Create a filtered dataframe using .isin() and a year condition


In [ ]:
# Count observations by lake in your filtered dataframe


---
## ◾Working with missing values

Missing values are common in environmental datasets. Instruments fail, samples are not collected, sensors are below detection limits, and historical datasets may combine many protocols or sampling campaigns.

In pandas, missing values are often represented as `NaN`.

Two common methods are:

- `.isna()` returns `True` for missing values
- `.notna()` returns `True` for non-missing values


In [ ]:
# Count missing values in each column
NTL_phys_data.isna().sum()

In [ ]:
# Calculate percent missing in each column
missing_percent = NTL_phys_data.isna().mean().sort_values(ascending=False) * 100
missing_percent

In [ ]:
# Select rows where dissolved oxygen is missing
NTL_missing_DO = NTL_phys_data.loc[NTL_phys_data["dissolvedOxygen"].isna()]
NTL_missing_DO.head()

In [ ]:
# Select rows where dissolved oxygen is not missing
NTL_complete_DO = NTL_phys_data.loc[NTL_phys_data["dissolvedOxygen"].notna()]
NTL_complete_DO.head()

In [ ]:
# Compare row counts
NTL_phys_data.shape[0], NTL_missing_DO.shape[0], NTL_complete_DO.shape[0]

---
### Exercise 4: Missing values

Answer the following questions.

1. Which column has the most missing values?
2. What percentage of `dissolvedOxygen` values are missing?
3. Create a dataframe containing only rows where `temperature`, `dissolvedOxygen`, and `depth` are all non-missing.


In [ ]:
# 1. Which column has the most missing values?


In [ ]:
# 2. What percentage of dissolvedOxygen values are missing?


In [ ]:
# 3. Keep rows where temperature, dissolvedOxygen, and depth are all non-missing


---
## A note on method chaining

As your pandas workflows become more complex, you may see code written as a **method chain**. Method chaining applies several steps in sequence while keeping the code readable.

For example, this chain filters the data to surface observations from three selected lakes and then resets the index.


In [ ]:
selected_lakes = ["Peter Lake", "Paul Lake", "Tuesday Lake"]

NTL_three_lakes_surface = (
    NTL_phys_data
    .loc[lambda df: df["lakename"].isin(selected_lakes)]
    .loc[lambda df: df["depth"] == 0]
    .reset_index(drop=True)
)

NTL_three_lakes_surface.head()

Method chaining is not required, but it is a useful pattern for building readable and reproducible data workflows.


---
## Capstone exercise: Create an analysis-ready subset

Create a new dataframe named `NTL_analysis_subset` that contains:

- Surface observations only (`depth == 0`)
- Peter Lake, Paul Lake, and Tuesday Lake only
- Non-missing `temperature` values
- Non-missing `dissolvedOxygen` values

Then answer:

1. How many rows are in the final subset?
2. How many observations are available for each lake?
3. What years are represented in the final subset?
4. Why might this subset be useful for exploratory analysis?


In [ ]:
# Create NTL_analysis_subset here


In [ ]:
# 1. Number of rows in the final subset


In [ ]:
# 2. Number of observations by lake

In [ ]:
# 3. Years represented in the final subset

**Discussion:** Briefly interpret the results before continuing.

---
## Key takeaways

- Data wrangling prepares raw data for exploration, visualization, and analysis.
- Boolean masks are `True`/`False` filters that can be used with `.loc[]`.
- `.query()` provides a readable alternative for row filtering.
- Use comparison operators such as `==`, `!=`, `<`, and `>=` to define conditions.
- Use logical operators to combine conditions: `&`, `|`, and `~` with Boolean masks; `and`, `or`, and `not` inside `.query()`.
- Use `.isin()` to filter by membership in a list of values.
- Use `.isna()` and `.notna()` to identify and filter missing values.


---
## Preview: next wrangling topics

In later wrangling lessons, we will extend these ideas to:

- Selecting columns
- Renaming variables
- Creating new variables
- Sorting records
- Grouped summaries
- Joining datasets
- Reshaping data between wide and long formats
